# 🌾 KisanMitra AI — Notebook 01: Data Collection & Preparation

**Author:** B.Tech AI/ML Project  
**Target Crops:** Potato & Tomato  
**Objective:** Verify 13 PlantVillage disease classes, perform stratified 70/15/15 Train/Val/Test splits, fetch Open-Meteo weather, and prepare Agmarknet mandi price series.

This notebook is structured to run smoothly in **Google Colab**, **Kaggle Notebooks**, or **Locally**.

## 1. Environment Setup & Repository Navigation

In [ ]:
# Check runtime environment
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/input')

print(f"Runtime: {'Google Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local Machine'}")

# If running in Colab, clone the repo if not already present
if IN_COLAB and not os.path.exists('Aikissan'):
    !git clone https://github.com/namangoyal176-10/Aikissan.git
    %cd Aikissan
    sys.path.insert(0, os.getcwd())
else:
    # Local / Kaggle project root setup
    project_root = Path.cwd().resolve()
    if project_root.name == 'notebooks':
        os.chdir(project_root.parent)
    sys.path.insert(0, str(Path.cwd()))

print("Working directory:", os.getcwd())

## 2. PlantVillage Dataset: Downloading & Folder Verification

We use **13 specific classes** from the PlantVillage dataset:
- **Potato (3 classes):** Early blight, Late blight, Healthy
- **Tomato (10 classes):** Bacterial spot, Early blight, Late blight, Leaf mold, Septoria leaf spot, Spider mites, Target spot, Yellow leaf curl virus, Mosaic virus, Healthy

In [ ]:
# In Kaggle, PlantVillage can be added via the 'Add Data' sidebar (dataset: emmarex/plantdisease or abdallahalidev/plantvillage-dataset)
# In Colab, you can download via opendatasets or kaggle API:

raw_plantvillage_dir = Path("data/raw/plantvillage")
raw_plantvillage_dir.mkdir(parents=True, exist_ok=True)

# Example Kaggle API download (uncomment in Colab if kaggle.json is uploaded):
# !pip install -q kaggle
# !kaggle datasets download -d emmarex/plantdisease -p data/raw/ --unzip

print("Target raw directory:", raw_plantvillage_dir.resolve())

## 3. Run Stratified 70/15/15 Split & Imbalance Analysis

This script:
1. Verifies that all 13 class folders exist.
2. Scans valid images and computes class distributions.
3. Analyzes class imbalance ratio.
4. Splits images into 70% Train, 15% Validation, and 15% Test.
5. Saves file lists to `data/processed/` (does NOT duplicate image files to save disk space).

In [ ]:
# Execute the modular verification and dataset preparation script
!python3 src/disease/prepare_dataset.py --data-dir data/raw/plantvillage

## 4. Mandi Agro-Weather Ingestion (Open-Meteo API)

Fetches daily historical mean temperature, humidity, and rainfall for our 3 target APMC mandis:
1. **Azadpur (Delhi)**
2. **Kolar (Karnataka)**
3. **Agra (Uttar Pradesh)**

*Open-Meteo Archive API is 100% free for educational/research use and requires no API key.*

In [ ]:
# Fetch past 2 years of daily weather observations for the 3 mandis
!python3 src/price/fetch_weather.py --start-date 2022-01-01 --end-date 2024-01-01

## 5. Agmarknet Mandi Price Ingestion & Standardization

We strictly adhere to a **Zero-Fake-Data policy**:
- Place real Agmarknet CSVs inside `data/raw/agmarknet/` (downloaded from https://agmarknet.gov.in)
- OR set `DATA_GOV_IN_API_KEY` in `.env`

In [ ]:
# Ingest and standardize Agmarknet records
!python3 src/price/fetch_agmarknet.py

## 6. Merge Prices with Weather & Generate Data Quality Report

In [ ]:
# Merge datasets on (date, mandi) and print data quality report
!python3 src/price/merge_data.py

## 7. Next Steps for Phase 3

With the dataset manifests and weather/price tables prepared in `data/processed/`, we are ready to move to **Phase 3: Disease Detection Model Training (MobileNetV3 / EfficientNet with Grad-CAM)**.